In [3]:
from decord import VideoReader
from decord import cpu, gpu
video_path = '/data2/local_datasets/ActivityNet/videos/v_mPtCJg-j4SM.mp4'
vr = VideoReader(video_path, ctx=cpu(0))

print('video frames:', len(vr))
# 1. the simplest way is to directly access frames
all_index = [1, 504, 625, 919, 1204, 1431, 1862, 2036, 2138, 2542, 2824, 3130, 3414, 3624, 3823,4245]
frames2 = vr.get_batch(all_index).asnumpy()
# frames2 = vr.get_batch(range(len(vr))).asnumpy()

# del frames2

video frames: 4594


In [1]:
vr.get_avg_fps()


NameError: name 'vr' is not defined

In [5]:
import av

video_path = '/data2/local_datasets/ActivityNet/videos/v_mPtCJg-j4SM.mp4'
container = av.open(video_path)

# 비디오 스트림을 선택합니다
video_stream = container.streams.video[0]

# 전체 프레임 수를 구하기 위해 duration과 time_base를 활용합니다.
num_frames = int(video_stream.duration * video_stream.time_base.denominator / video_stream.time_base.numerator)
print('video frames:', num_frames)

all_index = [1, 504, 625, 919, 1204, 1431, 1862, 2036, 2138, 2542, 2824, 3130, 3414, 3624, 3823, 4245]
frames2 = []

# 프레임을 가져옵니다
for i, frame in enumerate(container.decode(video_stream)):
    if i in all_index:
        frames2.append(frame.to_image())

container.close()


video frames: 1552025430000


In [3]:
import av

video_path = '/data2/local_datasets/ActivityNet/videos/v_6gzU9P-5tqE.mkv'
video_path = '/data2/local_datasets/ActivityNet/videos/v__tRAypMWUdc.webm'
container = av.open(video_path)
video_stream = container.streams.video[0]
total_frames = video_stream.frames
total_frames

0

In [5]:
import av

def load_from_file(
    filename,
    any_frame=False,
    backward=True
):

    container = av.open(filename)
    stream = container.streams.video[0]  # 1つ目のvideo stream．普通は1つしかないからこれでOK

    print("filename:", container.name)
    # print("filesize [bytes]:", container.size)
    print("filesize [kB]:", container.size // 1024)
    if container.size > 1024 * 1024:
        print("filesize [MB]:", container.size // 1024 // 1024)
    # print("bit_rate [b/s]:", container.bit_rate)
    print("bit_rate [kb/s]:", float(container.bit_rate) / 1024)
    print("container duration [sec]:",
          float(container.duration) / av.time_base)
    print("stream duration:", stream.duration)  # 謎
    print("frames:", stream.frames)
    print("guessed frames:",
          int(float(container.duration) / av.time_base * stream.base_rate))
    print("container format name:", container.format.name)
    print("container format long name:", container.format.long_name)
    print("codec name:", stream.codec_context.codec.name)
    print("codec long name:", stream.codec_context.codec.long_name)
    print("codec tag:", stream.codec_context.codec_tag)
    for md_str in container.metadata.keys():
        print(f"container metadata {md_str}:", container.metadata[md_str])
    print("base_rate [fps]:", stream.base_rate)
    print("rate [fps]:", stream.codec_context.rate)
    print("width [pix]:", stream.codec_context.width)
    print("height [pix]:", stream.codec_context.height)

    sec = 2  # 2秒時点へシーク
    container.seek(
        offset=sec // stream.time_base,
        any_frame=any_frame,
        backward=backward,
        stream=stream)

    for i, frame in enumerate(container.decode(video=0)):

        print(f"i:{i:3d}, time:{frame.time:.3f}, pts:{frame.pts}, "
              f"{frame.width}x{frame.height}, {frame.format.name} ",
              end="")
        if frame.key_frame:
            print("keyframe", end="")
        print()


        # ffmpegの機能でリサイズ．おそらく高速．
        frame = frame.reformat(width=244, height=244)

        # 以下ndarrayへの変換
        img = frame.to_ndarray()  # このままだと1チャンネルndarray．多分Yチャンネルのgray scale（未確認．コーデックによってはエラー発生）
        img = frame.to_rgb().to_ndarray()  # 3チャンネルndarray（RGB）
        img = frame.to_ndarray(format="rgb24")  # 同上
        img = frame.to_rgb().to_ndarray(width=244, height=244)  # リサイズしながらRGBへ変換
        img = frame.to_ndarray(format="rgb24", width=244, height=244)  # 同上

        # 以下PILへの変換
        img = frame.to_image()  # PIL image
        img = frame.to_image(width=244, height=244)  # リサイズしながらPILへ変換
        img.save(
            "frames.{:04d}.jpg".format(i),
            quality=80,
        )  # PIL imageの保存


        if i > 5:
            break


if __name__ == '__main__':

    filenames = [
        "/data2/local_datasets/ActivityNet/videos/v__tRAypMWUdc.webm",
        "/data2/local_datasets/ActivityNet/videos/v_6gzU9P-5tqE.mkv",
        "/data2/local_datasets/ActivityNet/videos/v__4LZrf1GL1s.mp4"
    ]

    for filename in filenames:
        print("==================")
        try:
            load_from_file(filename, any_frame=False, backward=True)
        except BaseException:
            continue


filename: /data2/local_datasets/ActivityNet/videos/v__tRAypMWUdc.webm
filesize [kB]: 94388
filesize [MB]: 92
bit_rate [kb/s]: 3502.6171875
container duration [sec]: 215.584
stream duration: None
frames: 0
guessed frames: 5174
container format name: matroska,webm
container format long name: Matroska / WebM
codec name: vp9
codec long name: Google VP9
codec tag:     
container metadata encoder: Lavf56.25.101
base_rate [fps]: 24
rate [fps]: None
width [pix]: 1920
height [pix]: 1080
i:  0, time:0.000, pts:0, 1920x1080, yuv420p keyframe
i:  1, time:0.042, pts:42, 1920x1080, yuv420p 
i:  2, time:0.083, pts:83, 1920x1080, yuv420p 
i:  3, time:0.125, pts:125, 1920x1080, yuv420p 
i:  4, time:0.167, pts:167, 1920x1080, yuv420p 
i:  5, time:0.208, pts:208, 1920x1080, yuv420p 
i:  6, time:0.250, pts:250, 1920x1080, yuv420p 
filename: /data2/local_datasets/ActivityNet/videos/v_6gzU9P-5tqE.mkv
filesize [kB]: 9040
filesize [MB]: 8
bit_rate [kb/s]: 796.1904296875
container duration [sec]: 90.836
stream